# 조건부확률: 정보로 원시 결과를 거르기

이 실습에서는 가중된 유한 원시 결과에서 조건부확률을 구현하고, 같은 교집합을 서로 다른 정보 조건으로 나눌 때 남는 상태가 어떻게 달라지는지 확인합니다. 마지막에는 가족 자체의 사건과 무작위 관측이 만들어 낸 기록을 구별합니다.

각 순서대로 구현한 뒤 fixture와 검사 셀을 실행하고, 실제 출력에 근거해 해석을 작성하세요.

In [17]:
import numpy as np


## 조건부확률을 가중 원시 결과에서 계산하기

### 왜 필요한가

운영 로그나 실험 결과는 원시 상태마다 같은 빈도로 발생하지 않을 수 있습니다. 이미 정보 `condition`을 관측했다면 그 조건과 양립하는 상태만 남기고, 그 안에서 `event`의 확률을 다시 계산해야 합니다.

### 먼저 생각할 점

`event`가 분자 쪽 질문이고 `condition`이 `|` 오른쪽의 관측 정보라면, 어느 집합이 필터 역할을 하는지 먼저 적어 보세요.

condition이 필터 역할을 해야 한다.

### 주어진 규칙과 작은 예

- `conditional_probability`는 `condition`에 속한 원시 결과의 총 가중치로, `event`와 `condition`의 교집합 가중치를 나눈 `P(event | condition)`을 반환해야 합니다.
- `weights`는 비어 있지 않고 음이 아닌 값만 가지며 합이 `1.0`이어야 합니다. `event`와 `condition`은 모두 `weights`의 key만 사용해야 합니다.
- `condition`의 가중치 합이 `0`이면 `ValueError`를 발생시켜야 합니다.

`{"s1": 1/8, "s2": 3/8, "s3": 1/4, "s4": 1/4}`에서 `event={"s1", "s2"}`, `condition={"s1", "s3"}`이면, 관측 뒤 남는 것은 `s1`, `s3`이고 분자는 `s1`의 가중치입니다.

<details><summary>힌트 1</summary>

먼저 condition의 가중치 합을 구한 뒤, event와 condition을 동시에 만족하는 원시 결과의 가중치 합을 구하세요.

</details>

<details><summary>힌트 2</summary>

교집합은 `event & condition`으로 만들 수 있습니다. 0인 condition 질량은 나누기 전에 거부해야 합니다.

</details>

### 해보기

아래 함수의 한 TODO만 채우세요. 제공된 입력 검사는 바꾸지 마세요.

### 확인하기

fixture를 실행한 뒤 `check_e01()`을 실행하세요. 정상 결과, 빈 event, 확률이 0인 condition을 각각 확인합니다.

### 정리하기

아래 빈칸에 2~4문장으로, 왜 `condition`이 남기는 상태를 결정하고 전체 가중치 합 대신 남은 질량으로 재정규화하는지 설명하세요.

In [18]:
def conditional_probability(weights: dict[str, float], event: set[str], condition: set[str]) -> float:
    """Return P(event | condition) in one finite weighted sample space."""
    if not weights or any(weight < 0 for weight in weights.values()):
        raise ValueError("weights must be nonempty and nonnegative")
    if not np.isclose(sum(weights.values()), 1.0):
        raise ValueError("weights must sum to 1.0")
    if not event.issubset(weights) or not condition.issubset(weights):
        raise ValueError("event and condition must use known outcomes")

    condition_mass = sum(weights[outcome] for outcome in condition)
    if np.isclose(condition_mass, 0.0):
        raise ValueError("condition must have positive probability")

    # TODO: event와 condition을 함께 만족하는 가중치를 합산한 뒤 condition_mass로 나누세요.
    joint_mass = sum(weights[outcome] for outcome in (event & condition))
    return float(joint_mass / condition_mass)


In [19]:
weights = {"s1": 1 / 8, "s2": 3 / 8, "s3": 1 / 4, "s4": 1 / 4}
overloaded = {"s1", "s2"}
alert = {"s1", "s3"}
print(conditional_probability(weights, overloaded, alert))


0.3333333333333333


In [20]:
def check_e01() -> None:
    weights = {"s1": 1 / 8, "s2": 3 / 8, "s3": 1 / 4, "s4": 1 / 4}
    np.testing.assert_allclose(conditional_probability(weights, {"s1", "s2"}, {"s1", "s3"}), 1 / 3)
    np.testing.assert_allclose(conditional_probability(weights, set(), {"s1", "s3"}), 0.0)

    try:
        conditional_probability({"seen": 1.0, "impossible": 0.0}, {"seen"}, {"impossible"})
    except ValueError:
        pass
    else:
        raise AssertionError("확률이 0인 condition은 ValueError여야 합니다")


check_e01()


### 결과 해석

<!-- TODO: condition이 필터가 되는 이유와 재정규화의 의미를 2~4문장으로 설명하세요. -->
condition은 이미 관측한 정보이므로, condition과 양립하는 원시 결과만 남긴다.
이 예에서는 s1과 s3이 남고, event에도 속하는 s1의 가중치를 남은 전체 가중치로 나누어 P(event | condition)을 계산한다.
전체 가중치 합에는 condition과 양립하지 않는 s2, s4도 포함되므로 분모로 쓰지 않으며, 남은 가중치를 다시 합 1이 되도록 재정규화 한다.

## 같은 교집합을 두 방향으로 계산하기

### 왜 필요한가

관측된 경보 기록에서 과부하를 추정하는 질문과, 과부하 기록에서 경보 발생을 측정하는 질문은 같은 교집합을 사용해도 서로 다릅니다.

### 먼저 생각할 점

`P(O | F)`와 `P(F | O)`에서 각각 `|` 오른쪽이 남기는 상태 집합을 먼저 적어 보세요.'


### 주어진 규칙과 작은 예

- `service_direction_probabilities`는 `(P(overloaded | alert), P(alert | overloaded))`를 이 순서로 반환해야 합니다.
- `overloaded`는 `s1`, `s2`이고 `alert`는 `s1`, `s3`입니다. 첫 계산은 `s1`, `s3`을 남기며, 둘째 계산은 `s1`, `s2`를 남깁니다.

가중치는 E01 fixture와 같습니다. 두 결과가 다른 것은 시간 순서가 아니라 관측 정보가 다르기 때문입니다.

<details><summary>힌트 1</summary>

첫 호출의 세 번째 인자는 `alert`이고, 둘째 호출의 세 번째 인자는 `overloaded`입니다.

</details>

<details><summary>힌트 2</summary>

두 호출은 같은 `weights`와 같은 두 사건을 쓰지만, event와 condition의 위치를 서로 바꿉니다.

</details>

### 해보기

반환 tuple의 두 빈 값을 조건부확률 호출로 채우세요.

### 확인하기

fixture와 `check_e02()`로 두 값의 순서와 서로 다른지를 확인하세요.

### 정리하기

아래 빈칸에, 두 분모가 각각 무엇을 뜻하는지와 `|`의 방향이 인과 방향이 아닌 이유를 2~4문장으로 작성하세요.

In [21]:
def service_direction_probabilities() -> tuple[float, float]:
    """Return overload-given-alert and alert-given-overload, in that order."""
    weights = {"s1": 1 / 8, "s2": 3 / 8, "s3": 1 / 4, "s4": 1 / 4}
    overloaded = {"s1", "s2"}
    alert = {"s1", "s3"}

    # TODO: alert를 조건으로 한 값과 overloaded를 조건으로 한 값을 순서대로 계산하세요.
    overload_given_alert = conditional_probability(weights, overloaded, alert)
    alert_given_overload = conditional_probability(weights, alert, overloaded)
    return float(overload_given_alert), float(alert_given_overload)


In [22]:
overload_given_alert, alert_given_overload = service_direction_probabilities()
print(overload_given_alert, alert_given_overload)


0.3333333333333333 0.25


In [23]:
def check_e02() -> None:
    overload_given_alert, alert_given_overload = service_direction_probabilities()
    np.testing.assert_allclose(overload_given_alert, 1 / 3)
    np.testing.assert_allclose(alert_given_overload, 1 / 4)
    np.testing.assert_equal(overload_given_alert == alert_given_overload, False)


check_e02()


### 결과 해석

<!-- TODO: 두 분모가 나타내는 관측 정보와 조건 방향이 인과 방향이 아닌 이유를 2~4문장으로 설명하세요. -->
P(overloaded | alert)의 분모는 경보가 울린 기록 전체의 확률이며, 이 조건에서는 s1과 s3만 남는다. 그중 과부하 상태는 s1뿐이므로 결과는 1/3이다.

P(alert | overloaded)의 분모는 과부하였던 기록 전체의 확률이며, 이 조건에서는 s1과 s2만 남는다. 그중 경보가 울린 상태는 s1뿐이므로 결과는 1/4이다.

두 값이 다른 이유는 | 오른쪽의 관측 정보가 서로 다른 기록 집합을 남기기 때문이다. 이는 경보와 과부하 중 무엇이 무엇을 일으켰는지에 대한 결론은 아니다.

## 사건 보고와 무작위 관측을 구별하기

### 왜 필요한가

‘적어도 한 명이 여자’라는 가족 사건과 ‘무작위로 만난 아이가 여자’라는 관측은 같은 정보가 아닙니다. 후자는 가족에서 아이를 고르는 과정을 통해 기록이 생성됩니다.

### 먼저 생각할 점

`GG`, `GB`, `BG`, `BB` 가족에서 아이 한 명을 균등하게 고를 때, 각 가족이 여자 관측을 만들어 낼 확률을 예측하세요.

### 주어진 규칙과 작은 예

이 실습의 국소 가정은 다음과 같습니다. 첫째·둘째는 구분되고, 각 아이의 성별은 서로 독립이며 여자·남자가 같은 확률입니다. 무작위 관측은 두 아이 중 한 명을 각각 같은 확률로 고르며, 누구를 고를지는 성별이나 가족 유형에 따라 달라지지 않습니다.

- `girl_observation_likelihood`는 family가 `GG`, `GB`, `BG`, `BB` 중 하나일 때, 그 가족에서 무작위로 고른 아이가 여자일 확률을 반환해야 합니다.
- family가 이 네 문자열 중 하나가 아니면 `ValueError`를 발생시켜야 합니다.
- 결과 해석에서는 ‘첫째가 여자’가 남기는 `GG`, `GB`와 ‘적어도 한 명이 여자’가 남기는 `GG`, `GB`, `BG`를 비교하고, 무작위 여자 관측에서 `GG`의 가중치가 prior family probability와 그 가족에서 여자를 관측할 확률의 곱이 되는 이유를 위 가정과 함께 설명해야 합니다.

가족 자체의 사전확률이 모두 `1/4`라면 여자 관측을 만들어 내는 확률은 `GG`에서 `1`, `GB`와 `BG`에서 `1/2`, `BB`에서 `0`입니다.

<details><summary>힌트 1</summary>

문자열 안의 `G` 개수는 그 가족에서 여자 아이를 고를 수 있는 위치 수입니다. 전체 위치 수는 둘입니다.

</details>

<details><summary>힌트 2</summary>

각 family에서 여자 아이가 놓인 위치 수와, 균등하게 선택될 수 있는 전체 위치 수를 이용해 비율을 구성하세요. 먼저 허용된 family인지 검사하세요.

</details>

### 해보기

허용된 family 검사를 유지하고 TODO의 likelihood 계산만 완성하세요.

### 확인하기

fixture와 `check_e03()`로 네 likelihood, 관측 뒤 `GG`의 비율, 잘못된 family를 확인하세요.

### 정리하기

아래 빈칸에 위 가정, 두 사건이 남기는 원시 결과, 그리고 무작위 여자 관측의 가중치가 왜 다른지를 3~5문장으로 작성하세요.

In [24]:
def girl_observation_likelihood(family: str) -> float:
    """Return the chance that a uniformly selected child from family is a girl."""
    if family not in {"GG", "GB", "BG", "BB"}:
        raise ValueError("family must be one of GG, GB, BG, BB")

    # TODO: 무작위로 한 아이를 고를 때 여자 관측을 만들 확률을 계산하세요.
    likelihood = 0
    n = len(family)

    for group in family:
        likelihood += group.count("G") / n


    return float(likelihood)


In [25]:
priors = {family: 1 / 4 for family in ("GG", "GB", "BG", "BB")}
girl_observation_weights = {family: priors[family] * girl_observation_likelihood(family) for family in priors}
girl_observation_mass = sum(girl_observation_weights.values())
print(girl_observation_weights)
print(girl_observation_weights["GG"] / girl_observation_mass)


{'GG': 0.25, 'GB': 0.125, 'BG': 0.125, 'BB': 0.0}
0.5


In [26]:
def check_e03() -> None:
    values = [girl_observation_likelihood(family) for family in ("GG", "GB", "BG", "BB")]
    np.testing.assert_allclose(values, [1.0, 0.5, 0.5, 0.0])

    priors = {family: 1 / 4 for family in ("GG", "GB", "BG", "BB")}
    weights = {family: priors[family] * girl_observation_likelihood(family) for family in priors}
    np.testing.assert_allclose(weights["GG"] / sum(weights.values()), 1 / 2)

    try:
        girl_observation_likelihood("GX")
    except ValueError:
        pass
    else:
        raise AssertionError("알 수 없는 family는 ValueError여야 합니다")


check_e03()


### 결과 해석

<!-- TODO: 국소 가정과 두 사건의 retained outcomes, 무작위 여자 관측의 가중치 차이를 3~5문장으로 설명하세요. -->
첫째·둘째가 구분되고, 각 성별은 독립이며 같은 확률이라는 가정에서
‘첫째가 여자’는 GG와 GB를 남기지만, ‘적어도 한 명이 여자’는 GG, GB, BG를 남긴다.

무작위로 만난 아이가 여자였다는 관측은 가족 사건이 아니라
가족에서 한 아이를 균등하게 고르는 수집 과정의 결과다.

그래서 GG는 여자 아이를 관측할 확률이 1이고, GB와 BG는 1/2이므로
사전 가족 확률에 이 관측 likelihood를 곱해 가중치를 계산해야 한다.